# Repository edition: 03_donor_followups

Run from this repository in your existing GPU environment. Source is in `../suites/03_donor_followups/`. The launcher creates a versioned working copy. No embedded base64 code or installation is needed. Review configuration before launching; optional stages are not all enabled by default. See `../docs/EXPERIMENTS.md` for paper mapping.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
SUPPLEMENTARY = next((p for p in _candidates if (p / "tools" / "notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    SUPPLEMENTARY = next((p / "supplementary" for p in _candidates if (p / "supplementary/tools/notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    raise FileNotFoundError("Open this notebook from inside the extracted supplementary repository.")
sys.path.insert(0, str(SUPPLEMENTARY / "tools"))
from notebook_sources import source_bundle as _source_bundle


# Three Ways — full follow-up experiments

Upload this notebook to your Lambda machine, alongside `causal_v1_results.zip`, or run it where your old `threeways_causal_results/causal_v1` directory exists. Run the cells in order. **All source is embedded; no Git clone or placeholder cohort files are needed.**

Defaults use the **entire eligible evaluation cohort** in your results, the exact original model revision, all four attention/gate combinations under two normalization policies, three edit-search modes with a 256-operation budget, and three random-search seeds. This is the real experiment; the short implementation checks are a separate prerequisite.

Included: contrastive origins, label-shift audit, diagnostics on correct cases, matched comparisons and repair responses, frozen hybrids, verified edit-search curves, full-vocabulary sink vectors and WikiText-103 frequency alignment. The notebook automatically builds corpus counts because you do not already have them.

**One unavoidable limitation:** experiment 4 needs actual pretraining gradients. Its code and basis builder are included, but that stage remains explicitly unavailable without those data. A released checkpoint cannot reconstruct them. WikiText frequency is a proxy, not the model's true training distribution.

Frozen-program interventions do not alone prove a unique causal failure stage. Edit counts are verified greedy upper bounds, not minimum counts or bits. Read the bundled README before interpreting the figures.

In [ ]:
# 1. Settings. No example limit: this runs the full eligible cohort.
RUN_NAME = "followup_v1"
INPUT_RESULTS = None  # auto-detect; or set the actual ZIP/directory path on Lambda
TORCH_CUDA_WHEELS = "cu128"  # change to cu126 only if required by your driver
RUN_IN_BACKGROUND = False  # True returns after launching; logs remain on disk
BUILD_WIKITEXT_REFERENCE = True  # full WikiText-103 train split, resumable counting
# If using your own frequency file, set BUILD_WIKITEXT_REFERENCE=False and override frequency_file.
OVERRIDES = {}
# Examples: {"edit_budget": 512} or {"subspace_manifest": "/actual/path/bases/manifest.json"}
# Changing code/configuration/input requires a new RUN_NAME. Do not set max_examples for the full run.


## 2. Prepare source
The launcher packages the visible source files in this repository into a versioned working copy. The cell verifies its checksum and extracts it into a new versioned directory.

In [ ]:
import base64, hashlib, io, json, os, pathlib, subprocess, sys, zipfile
WORK = pathlib.Path.cwd().resolve()
BUNDLE = _source_bundle('03_donor_followups')
EXPECTED_SHA256 = hashlib.sha256(base64.b64decode(_source_bundle('03_donor_followups'))).hexdigest()
raw = base64.b64decode(BUNDLE)
assert hashlib.sha256(raw).hexdigest() == EXPECTED_SHA256
CODE = WORK / ("threeways_followup_code_" + EXPECTED_SHA256[:12])
CODE.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        target = (CODE / name).resolve()
        if CODE.resolve() not in target.parents:
            raise RuntimeError("Unsafe archive path")
        if target.exists() and target.read_bytes() != z.read(name):
            raise RuntimeError("Extracted source has local edits. Preserve them and use a new directory.")
    z.extractall(CODE)
print("Source:", CODE)
print("Guide:", CODE / "README.txt")


## 3. Environment and kernel
Creates an isolated environment without changing the old causal environment. Experiments use its Python directly; no notebook kernel restart is required. The cell also registers **Python (Three Ways Follow-up)** for later use.

If system Python lacks `venv`, install `python3-venv` in the Lambda terminal and rerun. For 1B models, use at least 16 GB VRAM; larger models need more. Full vocabulary caches need disk space, and repeated edit searches can take substantial time.

In [ ]:
VENV = WORK / "threeways_followup_env"
PYTHON = VENV / "bin" / "python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
marker = VENV / "installed.json"
expected = {"bundle": EXPECTED_SHA256, "cuda_wheels": TORCH_CUDA_WHEELS}
if not marker.exists() or json.loads(marker.read_text()) != expected:
# Use the existing environment; dependencies are documented in requirements.txt.
# Use the existing environment; dependencies are documented in requirements.txt.
# Use the existing environment; dependencies are documented in requirements.txt.
    subprocess.run([str(PYTHON), "-m", "ipykernel", "install", "--user", "--name", "threeways-followup",
                    "--display-name", "Python (Three Ways Follow-up)"], check=True)
    marker.write_text(json.dumps(expected))
subprocess.run([str(PYTHON), "-c", "import torch,transformers; assert torch.cuda.is_available(), 'CUDA unavailable'; "
    "print('GPU:',torch.cuda.get_device_name(0)); print('Torch:',torch.__version__); "
    "print('Transformers:',transformers.__version__); print('VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,1))"], check=True)


## 4. Locate your previous results and authorize model download
Upload the actual results ZIP if it is not already on the machine. The code automatically locates the usual prior output directory. Your model SHA and input token IDs are read from that data.

Meta models require approved Hugging Face access. Leave the token blank to use a saved login. Tokens are not written to the configuration or result archive.

In [ ]:
if INPUT_RESULTS is None:
    candidates = [WORK / "causal_v1_results.zip", WORK / "threeways_causal_results" / "causal_v1",
                  pathlib.Path.home() / "causal_v1_results.zip",
                  pathlib.Path.home() / "threeways_causal_results" / "causal_v1"]
    INPUT = next((p for p in candidates if p.exists()), None)
    if INPUT is None:
        raise FileNotFoundError("Upload causal_v1_results.zip next to this notebook, or set INPUT_RESULTS to your existing results directory.")
else:
    INPUT = pathlib.Path(INPUT_RESULTS).expanduser().resolve()
    if not INPUT.exists():
        raise FileNotFoundError(INPUT)
print("Input:", INPUT)
from getpass import getpass
if not os.environ.get("HF_TOKEN"):
    token = getpass("Hugging Face token (Enter to use existing login): ").strip()
    if token:
        os.environ["HF_TOKEN"] = token
    del token


## 5. Verify the implementation
Tiny random models check the equations, numerical certificates, attention constraints, recovery/pruning, full runner, reports and resume. They are tests of the code, not the scientific results. The real full-cohort run is in cell 8.

In [ ]:
subprocess.run([str(PYTHON), "-m", "pytest", "tests", "-q"], cwd=CODE, check=True)


## 6. Build the independent frequency reference
By default this counts the **entire WikiText-103 training split**, using each exact model tokenizer, with a pinned dataset revision and resumable checkpoints. First use downloads the corpus; subsequent runs reuse the verified counts.

[WikiText-103](https://huggingface.co/datasets/Salesforce/wikitext) is an independently chosen corpus, not a reconstruction of Llama's pretraining mixture. That distinction is saved in the results. If this cell is interrupted, rerun it to resume. Disabling it still permits the sink-variation experiment, but frequency alignment is marked unavailable.

In [ ]:
REFERENCE = WORK / "threeways_followup_frequency" / EXPECTED_SHA256[:12]
FREQUENCY_FILES = {}
if BUILD_WIKITEXT_REFERENCE:
    subprocess.run([str(PYTHON), "-u", "-m", "followup.reference_corpus", "--input", str(INPUT),
                    "--out", str(REFERENCE)], cwd=CODE, check=True)
    FREQUENCY_FILES = json.loads((REFERENCE / "mapping.json").read_text())
print("Frequency files:", FREQUENCY_FILES)


## 7. Save configuration and run cached analyses
This first stage needs no model forward passes. It uses all eligible examples in the old results. The new GPU stages use the same configuration and output directory, so these results are retained.

No pretraining-gradient data were supplied: leave `subspace_manifest=None`. If you later obtain genuine training gradients, the README describes their schema and the included basis builder.

In [ ]:
CONFIG = json.loads(subprocess.check_output([str(PYTHON), "-c",
    "import json; from followup.common import DEFAULT; print(json.dumps(DEFAULT))"], cwd=CODE, text=True))
CONFIG["frequency_files"] = FREQUENCY_FILES
CONFIG.update(OVERRIDES)
RESULTS = WORK / "threeways_followup_results" / RUN_NAME
CONFIG_PATH = WORK / (RUN_NAME + "_followup_config.json")
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(CONFIG, indent=2))
print("Results:", RESULTS)
BASE_COMMAND = [str(PYTHON), "-u", "-m", "followup.run", "--input", str(INPUT),
                "--out", str(RESULTS), "--config", str(CONFIG_PATH)]
subprocess.run(BASE_COMMAND + ["--cpu-only"], cwd=CODE, check=True)


## 8. Run the full GPU follow-up
No downsampling is applied. The runner saves each completed stage/example/search and rejects concurrent writers. Rerun this cell with the **same settings** to resume. An interrupted individual edit search restarts that search, while completed searches are retained.

With `RUN_IN_BACKGROUND=True`, the cell returns after launch. Otherwise it streams the log and waits. Interrupting this cell leaves the worker running and logging to disk; do not launch a second copy. The monitor cell shows its PID and status.

In [ ]:
import time
RESULTS.mkdir(parents=True, exist_ok=True)
LOG = RESULTS / "execution.log"
PID_FILE = RESULTS / "notebook_worker.pid"
STATUS = RESULTS / "exit_code.txt"
if PID_FILE.exists():
    old_pid = int(PID_FILE.read_text())
    try:
        os.kill(old_pid, 0)
    except ProcessLookupError:
        PID_FILE.unlink()
    else:
        raise RuntimeError(f"Process {old_pid} may still be running. Use the monitor cell before launching another worker.")
STATUS.unlink(missing_ok=True)
supervisor = """import os,pathlib,subprocess,sys
pid=pathlib.Path(sys.argv[1]);status=pathlib.Path(sys.argv[2]);pid.write_text(str(os.getpid()))
try:
    result=subprocess.call(sys.argv[3:]);status.write_text(str(result))
finally:
    pid.unlink(missing_ok=True)
sys.exit(result)
"""
start_offset = LOG.stat().st_size if LOG.exists() else 0
with LOG.open("a") as logfile:
    worker = subprocess.Popen([str(PYTHON), "-c", supervisor, str(PID_FILE), str(STATUS), *BASE_COMMAND],
        cwd=CODE, stdout=logfile, stderr=subprocess.STDOUT, start_new_session=True)
print("Worker:", worker.pid, "Log:", LOG)
if not RUN_IN_BACKGROUND:
    try:
        with LOG.open() as log:
            log.seek(start_offset)
            while worker.poll() is None:
                chunk = log.read()
                if chunk:
                    print(chunk, end="", flush=True)
                time.sleep(1)
            print(log.read(), end="")
        if worker.returncode:
            raise RuntimeError(f"Run stopped with exit code {worker.returncode}. Inspect {LOG}; completed searches are saved.")
    except KeyboardInterrupt:
        print("Notebook monitoring interrupted; worker continues. PID:", worker.pid)


## 9. Monitor and inspect
A zero exit code means the enabled stages finished, including any explicitly recorded unavailable stages. Check coverage and missing external data before interpreting the figures. A positive fixed-competitor margin need not make gold top-1; neither metric establishes full-answer recovery.

In [ ]:
from collections import deque
from IPython.display import FileLink, display
if LOG.exists():
    with LOG.open() as f:
        print("".join(deque(f, maxlen=25)))
print("Worker PID:", PID_FILE.read_text() if PID_FILE.exists() else "none recorded")
print("Exit code:", STATUS.read_text() if STATUS.exists() else "not recorded yet")
for p in sorted(RESULTS.glob("*/*/external_data_status.json")):
    print(p.parent, p.read_text())
for p in sorted(RESULTS.glob("*/*/reports/*.pdf")):
    display(FileLink(str(p)))
for p in sorted(RESULTS.glob("*/*/cpu/*.pdf")):
    display(FileLink(str(p)))


In [ ]:
# Optional: regenerate reports after the worker finishes. No model weights loaded.
REBUILD_REPORTS = False
if REBUILD_REPORTS:
    subprocess.run(BASE_COMMAND + ["--report-only"], cwd=CODE, check=True)


## 10. Download results
The compact archive includes metrics, search traces, donor audits, certificates, configuration, exact source and frequency counts. It excludes large full-vocabulary vector caches, duplicated input ZIP contents and model weights. Keep `vectors/` on the machine for later sink reanalysis.

By default this cell refuses to label a still-running or failed run as complete. Set `ALLOW_PARTIAL=True` only if you intentionally want a partial snapshot.

In [ ]:
ALLOW_PARTIAL = False
complete = STATUS.exists() and STATUS.read_text().strip() == "0" and not PID_FILE.exists()
if not complete and not ALLOW_PARTIAL:
    raise RuntimeError("Run is not complete. Wait for the worker or inspect the log. Set ALLOW_PARTIAL=True only for an intentional partial export.")
ARCHIVE = WORK / (RUN_NAME + ("_results.zip" if complete else "_PARTIAL_results.zip"))
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.rglob("*")):
        relative = p.relative_to(RESULTS)
        if p.is_file() and not ({"vectors", "inputs"} & set(relative.parts)) and p.name != ".run.lock":
            z.write(p, relative)
    z.write(CONFIG_PATH, "configuration.json")
    for p in sorted(CODE.rglob("*")):
        if p.is_file() and not ({"__pycache__", ".pytest_cache"} & set(p.parts)):
            z.write(p, "source/" + str(p.relative_to(CODE)))
    frequency_paths = set(CONFIG.get("frequency_files", {}).values())
    if CONFIG.get("frequency_file"):
        frequency_paths.add(CONFIG["frequency_file"])
    for i, filename in enumerate(sorted(frequency_paths)):
        p = pathlib.Path(filename)
        z.write(p, f"frequency/{i}_{p.name}")
        z.write(p.with_suffix(".json"), f"frequency/{i}_{p.with_suffix('.json').name}")
print("Saved:", ARCHIVE)
display(FileLink(str(ARCHIVE)))
